# 03 — Stadtreferenz

## Zweck
Hier entsteht der **gemeinsame Stadtkatalog** des Projekts. Jede Stadt bekommt eine stabile
`city_id` (z. B. `vienna_at`). Alle späteren Notebooks (EEA, Wikipedia, Open-Meteo, Spark, Gold)
verknüpfen ihre Daten über diese `city_id`.

## Ausgabe
- `data/silver/city_reference.parquet` — `city_id`, Name, Land, Koordinaten
- `data/silver/city_reference.csv` — gut lesbare Kopie

## Konfiguration

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SILVER_DIR = PROJECT_ROOT / "data" / "silver"
SILVER_DIR.mkdir(parents=True, exist_ok=True)
print({"silver_dir": str(SILVER_DIR)})

{'silver_dir': '/workspace/data/silver'}


## Die acht Städte
Die `city_id` folgt dem Muster `<name>_<ländercode>`. Koordinaten dienen später dem Open-Meteo-Abruf,
die Wikipedia-URL dem Scraping. Bevölkerungswerte bleiben hier leer — sie kommen aus Notebook `05`.

In [2]:
city_reference_df = pd.DataFrame([
    {"city_id": "vienna_at",    "city_name": "Vienna",    "country_code": "AT", "latitude": 48.2082, "longitude": 16.3738, "wikipedia_url": "https://en.wikipedia.org/wiki/Vienna"},
    {"city_id": "berlin_de",    "city_name": "Berlin",    "country_code": "DE", "latitude": 52.5200, "longitude": 13.4050, "wikipedia_url": "https://en.wikipedia.org/wiki/Berlin"},
    {"city_id": "paris_fr",     "city_name": "Paris",     "country_code": "FR", "latitude": 48.8566, "longitude": 2.3522,  "wikipedia_url": "https://en.wikipedia.org/wiki/Paris"},
    {"city_id": "madrid_es",    "city_name": "Madrid",    "country_code": "ES", "latitude": 40.4168, "longitude": -3.7038, "wikipedia_url": "https://en.wikipedia.org/wiki/Madrid"},
    {"city_id": "rome_it",      "city_name": "Rome",      "country_code": "IT", "latitude": 41.9028, "longitude": 12.4964, "wikipedia_url": "https://en.wikipedia.org/wiki/Rome"},
    {"city_id": "amsterdam_nl", "city_name": "Amsterdam", "country_code": "NL", "latitude": 52.3676, "longitude": 4.9041,  "wikipedia_url": "https://en.wikipedia.org/wiki/Amsterdam"},
    {"city_id": "warsaw_pl",    "city_name": "Warsaw",    "country_code": "PL", "latitude": 52.2297, "longitude": 21.0122, "wikipedia_url": "https://en.wikipedia.org/wiki/Warsaw"},
    {"city_id": "prague_cz",    "city_name": "Prague",    "country_code": "CZ", "latitude": 50.0755, "longitude": 14.4378, "wikipedia_url": "https://en.wikipedia.org/wiki/Prague"},
])
city_reference_df

,city_id,city_name,country_code,latitude,longitude,wikipedia_url
0,vienna_at,Vienna,AT,48.2082,16.3738,https://en.wikipedia.org/wiki/Vienna
1,berlin_de,Berlin,DE,52.5200,13.4050,https://en.wikipedia.org/wiki/Berlin
2,paris_fr,Paris,FR,48.8566,2.3522,https://en.wikipedia.org/wiki/Paris
3,madrid_es,Madrid,ES,40.4168,-3.7038,https://en.wikipedia.org/wiki/Madrid
4,rome_it,Rome,IT,41.9028,12.4964,https://en.wikipedia.org/wiki/Rome
5,amsterdam_nl,Amsterdam,NL,52.3676,4.9041,https://en.wikipedia.org/wiki/Amsterdam
6,warsaw_pl,Warsaw,PL,52.2297,21.0122,https://en.wikipedia.org/wiki/Warsaw
7,prague_cz,Prague,CZ,50.0755,14.4378,https://en.wikipedia.org/wiki/Prague


## Validierung
Drei einfache Invarianten: 8 eindeutige Städte, gültige Koordinaten, keine fehlenden Pflichtfelder.

In [3]:
required = ["city_id", "city_name", "country_code", "latitude", "longitude"]
assert len(city_reference_df) == 8, "Es werden genau 8 Städte erwartet."
assert city_reference_df["city_id"].is_unique, "city_id muss eindeutig sein."
assert city_reference_df[required].notna().all().all(), "Pflichtfelder dürfen nicht leer sein."
assert city_reference_df["latitude"].between(-90, 90).all(), "Ungültiger Breitengrad."
assert city_reference_df["longitude"].between(-180, 180).all(), "Ungültiger Längengrad."
print("OK: alle Invarianten bestanden.")

OK: alle Invarianten bestanden.


## Speichern

In [4]:
parquet_path = SILVER_DIR / "city_reference.parquet"
csv_path = SILVER_DIR / "city_reference.csv"
city_reference_df.to_parquet(parquet_path, index=False)
city_reference_df.to_csv(csv_path, index=False)
print(f"Geschrieben: {parquet_path.name}, {csv_path.name}")
pd.read_parquet(parquet_path).head()

Geschrieben: city_reference.parquet, city_reference.csv


,city_id,city_name,country_code,latitude,longitude,wikipedia_url
0,vienna_at,Vienna,AT,48.2082,16.3738,https://en.wikipedia.org/wiki/Vienna
1,berlin_de,Berlin,DE,52.5200,13.4050,https://en.wikipedia.org/wiki/Berlin
2,paris_fr,Paris,FR,48.8566,2.3522,https://en.wikipedia.org/wiki/Paris
3,madrid_es,Madrid,ES,40.4168,-3.7038,https://en.wikipedia.org/wiki/Madrid
4,rome_it,Rome,IT,41.9028,12.4964,https://en.wikipedia.org/wiki/Rome


## Nächster Schritt
Notebook `04` ausführen — historische EEA-Messwerte abrufen.